In [10]:
import os
import random
from typing import List, Dict

import numpy as np
import pandas as pd

import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

# =========================
# Config (LOCAL / VS Code)
# =========================
BASE_DIR = "./add_feature_dataset/GES"
TRAIN_CSV = os.path.join(BASE_DIR, "data_with_features_GES_train.csv")
VAL_CSV   = os.path.join(BASE_DIR, "data_with_features_GES_val.csv")
TEST_CSV  = os.path.join(BASE_DIR, "data_with_features_GES_test.csv")

TARGET_CANDIDATES = ["label", "target", "y", "failure", "bank_failure", "default", "is_failed"]

# ✅ seed only changes (5 runs)
SEEDS = [35, 42, 55, 64, 100]

# ECE bins
ECE_BINS = 15

# =========================
# Output (CSV save)
# =========================
OUT_DIR = "./results_tables"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_RUN_CSV  = os.path.join(OUT_DIR, "results_F_GES_34edges_5seeds_runs.csv")
OUT_MEAN_CSV = os.path.join(OUT_DIR, "results_F_GES_34edges_5seeds_mean.csv")

# =========================
# MUST-USE EDGES (F-set only)
# - 정확히 이 34개만 사용 (다른 edge / original feature 절대 불가)
# =========================
EDGES_34 = [
  "edge_GES__asset_liabilities__rect_turn",
  "edge_GES__asset_liabilities__debt_ratio",
  "edge_GES__debt_ratio__rect_turn",
  "edge_GES__debt_ratio__leverage_ratio",
  "edge_GES__roa__asset_turnover",
  "edge_GES__asset_turnover__rect_turn",
  "edge_GES__roa__roe",
  "edge_GES__asset_liabilities__leverage_ratio",
  "edge_GES__debt_ratio2__capitalization_ratio",
  "edge_GES__debt_ratio__roa",
  "edge_GES__roa__rect_turn",
  "edge_GES__debt_ratio2__longtermdebt_invcap",
  "edge_GES__debt_ratio2__totaldebt_invcap",
  "edge_GES__debt_ebitda__debt_ratio2",
  "edge_GES__debt_ratio2__roa",
  "edge_GES__cash_debt__asset_liabilities",
  "edge_GES__roa__totaldebt_invcap",
  "edge_GES__asset_turnover__totaldebt_invcap",
  "edge_GES__asset_turnover__leverage_ratio",
  "edge_GES__roa__leverage_ratio",
  "edge_GES__cash_debt__debt_ratio2",
  "edge_GES__debt_ebitda__roa",
  "edge_GES__asset_turnover__roe",
  "edge_GES__debt_ratio2__leverage_ratio",
  "edge_GES__debt_ratio__capitalization_ratio",
  "edge_GES__asset_turnover__capitalization_ratio",
  "edge_GES__debt_ebitda__roe",
  "edge_GES__cash_debt__totaldebt_invcap",
  "edge_GES__debt_ratio2__debt_ratio",
  "edge_GES__asset_liabilities__roe",
  "edge_GES__debt_ratio__roe",
  "edge_GES__cash_debt__rect_turn",
  "edge_GES__cash_debt__debt_ratio",
  "edge_GES__cash_debt__longtermdebt_invcap",
]

# =========================
# HARD BLOCK: index-like remover
# =========================
def is_index_col_name(col: str) -> bool:
    c = str(col).strip()
    cl = c.lower()
    if cl.startswith("unnamed"):
        return True
    if cl in {"index", "_index"}:
        return True
    if cl.endswith("_index"):
        return True
    return False

def drop_indexlike_cols(df: pd.DataFrame, name: str) -> pd.DataFrame:
    drop_cols = [c for c in df.columns if is_index_col_name(c)]
    if drop_cols:
        print(f"[DROP] {name}: index-like -> {drop_cols}")
        df = df.drop(columns=drop_cols)
    bad = [c for c in df.columns if str(c).lower().startswith("unnamed")]
    if bad:
        raise RuntimeError(f"[FATAL] {name}: Unnamed columns still exist: {bad}")
    return df

def detect_target_col(df: pd.DataFrame) -> str:
    for c in TARGET_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(f"Target column not found among {TARGET_CANDIDATES}")

def ensure_exact_edges_only(df: pd.DataFrame, edge_cols: List[str], target_col: str, name: str):
    missing = [c for c in edge_cols if c not in df.columns]
    if missing:
        raise RuntimeError(f"[FATAL] {name}: Missing required edges: {missing}")
    if target_col not in df.columns:
        raise RuntimeError(f"[FATAL] {name}: target_col '{target_col}' not found.")

def set_seed_everywhere(seed: int):
    random.seed(seed)
    np.random.seed(seed)

# =========================
# Metrics
# =========================
def brier_score(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    y_true = y_true.astype(float)
    y_prob = np.clip(y_prob, 0.0, 1.0)
    return float(np.mean((y_prob - y_true) ** 2))

def expected_calibration_error(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 15) -> float:
    y_true = y_true.astype(int)
    y_prob = np.clip(y_prob, 0.0, 1.0)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    n = len(y_true)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (y_prob >= lo) & (y_prob < hi) if i < n_bins - 1 else (y_prob >= lo) & (y_prob <= hi)
        if not np.any(mask):
            continue
        acc = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += (mask.sum() / n) * abs(acc - conf)
    return float(ece)

def best_f1_threshold(y_true: np.ndarray, y_prob: np.ndarray, n_grid: int = 101) -> float:
    thresholds = np.linspace(0.0, 1.0, n_grid)
    best_t, best_f1 = 0.5, -1.0
    for t in thresholds:
        pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t)

def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray, thr: float) -> Dict[str, float]:
    y_pred = (y_prob >= thr).astype(int)
    return {
        "AUROC": float(roc_auc_score(y_true, y_prob)),
        "AUPRC": float(average_precision_score(y_true, y_prob)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
        "Brier": brier_score(y_true, y_prob),
        "ECE": expected_calibration_error(y_true, y_prob, n_bins=ECE_BINS),
        "THR": float(thr),
    }

def read_csv_safely(path: str, name: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"[FATAL] Missing file: {os.path.abspath(path)}")
    df = pd.read_csv(path, low_memory=False)
    df = drop_indexlike_cols(df, name)
    return df

# =========================
# Pretty table printer (your style)
# =========================
def print_results_tables(df_res: pd.DataFrame):
    # order columns exactly like your example (+ Brier, ECE)
    df_show = df_res[["AUROC", "AUPRC", "F1", "Brier", "ECE", "THR", "SEED"]].copy()

    for c in ["AUROC", "AUPRC", "F1", "Brier", "ECE", "THR"]:
        df_show[c] = df_show[c].astype(float)
    df_show["SEED"] = df_show["SEED"].astype(int)

    df_show = df_show.sort_values("SEED").reset_index(drop=True)

    pd.set_option("display.width", 160)
    pd.set_option("display.max_columns", 30)

    fmt = {
        "AUROC": "{:.6f}".format,
        "AUPRC": "{:.6f}".format,
        "F1": "{:.6f}".format,
        "Brier": "{:.6f}".format,
        "ECE": "{:.6f}".format,
        "THR": "{:.2f}".format,   # 0.01 / 0.06 처럼 보이게
        "SEED": "{:d}".format,
    }

    print("\n===== RESULTS (5 runs) =====")
    print(df_show.to_string(index=True, formatters=fmt))

    mean_row = {
        "AUROC": float(df_show["AUROC"].mean()),
        "AUPRC": float(df_show["AUPRC"].mean()),
        "F1": float(df_show["F1"].mean()),
        "Brier": float(df_show["Brier"].mean()),
        "ECE": float(df_show["ECE"].mean()),
        "THR": float(df_show["THR"].mean()),
    }

    print("\n===== MEAN =====")
    print(
        f"AUROC={mean_row['AUROC']:.6f} | "
        f"AUPRC={mean_row['AUPRC']:.6f} | "
        f"F1={mean_row['F1']:.6f} | "
        f"Brier={mean_row['Brier']:.6f} | "
        f"ECE={mean_row['ECE']:.6f} | "
        f"THR(mean)={mean_row['THR']:.4f}"
    )

    return df_show, mean_row

# =========================
# Main
# =========================
def main():
    # ---- path debug ----
    print("[PATH]")
    print(" TRAIN =", os.path.abspath(TRAIN_CSV))
    print(" VAL   =", os.path.abspath(VAL_CSV))
    print(" TEST  =", os.path.abspath(TEST_CSV))
    print(" OUT_DIR =", os.path.abspath(OUT_DIR))

    # load
    df_train = read_csv_safely(TRAIN_CSV, "TRAIN")
    df_val   = read_csv_safely(VAL_CSV,   "VAL")
    df_test  = read_csv_safely(TEST_CSV,  "TEST")

    target_col = detect_target_col(df_train)
    if target_col not in df_val.columns or target_col not in df_test.columns:
        raise RuntimeError(f"[FATAL] target_col '{target_col}' not found in val/test.")

    # enforce edges exist
    ensure_exact_edges_only(df_train, EDGES_34, target_col, "TRAIN")
    ensure_exact_edges_only(df_val,   EDGES_34, target_col, "VAL")
    ensure_exact_edges_only(df_test,  EDGES_34, target_col, "TEST")

    # F set: edges only
    X_train = df_train[EDGES_34].to_numpy(dtype=np.float32)
    X_val   = df_val[EDGES_34].to_numpy(dtype=np.float32)
    X_test  = df_test[EDGES_34].to_numpy(dtype=np.float32)

    y_train = df_train[target_col].to_numpy(dtype=np.int64)
    y_val   = df_val[target_col].to_numpy(dtype=np.int64)
    y_test  = df_test[target_col].to_numpy(dtype=np.int64)

    print(f"[OK] Loaded | n_train={len(df_train)} n_val={len(df_val)} n_test={len(df_test)}")
    print(f"[OK] Using F-set edges ONLY | n_feat={X_train.shape[1]} (must be 34)")

    # LightGBM CPU params (fixed)
    base_params = dict(
        n_estimators=2000,
        learning_rate=0.02,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=-1,
        verbose=-1,
    )

    results = []
    for i, seed in enumerate(SEEDS, 1):
        set_seed_everywhere(seed)

        clf = lgb.LGBMClassifier(**base_params, random_state=seed)
        clf.fit(X_train, y_train)

        # threshold from VAL (best F1)
        val_prob = clf.predict_proba(X_val)[:, 1]
        thr = best_f1_threshold(y_val, val_prob)

        # evaluate on TEST
        test_prob = clf.predict_proba(X_test)[:, 1]
        m = compute_metrics(y_test, test_prob, thr)
        m["SEED"] = int(seed)
        results.append(m)

        print(
            f"[RUN {i}/5] seed={seed} | "
            f"AUROC={m['AUROC']:.6f} AUPRC={m['AUPRC']:.6f} F1={m['F1']:.6f} "
            f"Brier={m['Brier']:.6f} ECE={m['ECE']:.6f} THR={m['THR']:.4f}"
        )

    df_res = pd.DataFrame(results)

    # pretty print + mean
    df_show, mean_row = print_results_tables(df_res)

    # =========================
    # Save CSVs
    # =========================
    # runs
    df_runs = df_show[["SEED", "AUROC", "AUPRC", "F1", "Brier", "ECE", "THR"]].copy()
    df_runs.to_csv(OUT_RUN_CSV, index=False)

    # mean
    df_mean = pd.DataFrame([{
        "SEED": "MEAN",
        "AUROC": mean_row["AUROC"],
        "AUPRC": mean_row["AUPRC"],
        "F1": mean_row["F1"],
        "Brier": mean_row["Brier"],
        "ECE": mean_row["ECE"],
        "THR": mean_row["THR"],
    }])
    df_mean.to_csv(OUT_MEAN_CSV, index=False)

    print(f"\n[SAVED] runs -> {os.path.abspath(OUT_RUN_CSV)}")
    print(f"[SAVED] mean -> {os.path.abspath(OUT_MEAN_CSV)}")

if __name__ == "__main__":
    main()


[PATH]
 TRAIN = d:\University\3-2.5\PADA_Lab\Bank_Failure_Prediction_3\9_best_model_no_weight\add_feature_dataset\GES\data_with_features_GES_train.csv
 VAL   = d:\University\3-2.5\PADA_Lab\Bank_Failure_Prediction_3\9_best_model_no_weight\add_feature_dataset\GES\data_with_features_GES_val.csv
 TEST  = d:\University\3-2.5\PADA_Lab\Bank_Failure_Prediction_3\9_best_model_no_weight\add_feature_dataset\GES\data_with_features_GES_test.csv
 OUT_DIR = d:\University\3-2.5\PADA_Lab\Bank_Failure_Prediction_3\9_best_model_no_weight\results_tables
[OK] Loaded | n_train=12516 n_val=1788 n_test=3577
[OK] Using F-set edges ONLY | n_feat=34 (must be 34)


c:\Users\User\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[RUN 1/5] seed=35 | AUROC=0.932920 AUPRC=0.462587 F1=0.464516 Brier=0.017036 ECE=0.016818 THR=0.0100


c:\Users\User\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[RUN 2/5] seed=42 | AUROC=0.933506 AUPRC=0.464367 F1=0.481203 Brier=0.017082 ECE=0.016775 THR=0.0600


...

c:\Users\User\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[RUN 4/5] seed=64 | AUROC=0.933868 AUPRC=0.457253 F1=0.437500 Brier=0.017206 ECE=0.017133 THR=0.0100


c:\Users\User\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[RUN 5/5] seed=100 | AUROC=0.933546 AUPRC=0.450033 F1=0.445860 Brier=0.017455 ECE=0.017454 THR=0.0100

===== RESULTS (5 runs) =====
     AUROC    AUPRC       F1    Brier      ECE  THR SEED
0 0.932920 0.462587 0.464516 0.017036 0.016818 0.01   35
1 0.933506 0.464367 0.481203 0.017082 0.016775 0.06   42
2 0.933615 0.454022 0.445860 0.017168 0.016958 0.01   55
3 0.933868 0.457253 0.437500 0.017206 0.017133 0.01   64
4 0.933546 0.450033 0.445860 0.017455 0.017454 0.01  100

===== MEAN =====
AUROC=0.933491 | AUPRC=0.457653 | F1=0.454988 | Brier=0.017189 | ECE=0.017028 | THR(mean)=0.0200

[SAVED] runs -> d:\University\3-2.5\PADA_Lab\Bank_Failure_Prediction_3\9_best_model_no_weight\results_tables\results_F_GES_34edges_5seeds_runs.csv
[SAVED] mean -> d:\University\3-2.5\PADA_Lab\Bank_Failure_Prediction_3\9_best_model_no_weight\results_tables\results_F_GES_34edges_5seeds_mean.csv


c:\Users\User\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
